# MMDP-OD-RTDP Comparison — Final Experiment

The experiment is fixed and requires no parameter choices:

- one map per difficulty level
- 1-6 agents
- Baseline RTDP vs OD-RTDP
- two paired seeds
- planning up to 60 seconds or until `solved`
- evaluation of up to 5 episodes, with a maximum budget of 8 seconds
- no diagnostics and no conflict-risk calculations
- transition cache bounded identically for both algorithms

All results are saved to a single file:

`/content/MMDP_OUTPUT/MMDP_results.csv`

The CSV contains the four key metrics: time, memory, number of states examined,
and success rate, together with basic run-identity fields. Nothing is
downloaded automatically.

In [ ]:
# Fetch the code from GitHub and prepare the working environment
import subprocess, sys
from pathlib import Path
import shutil

# Clone the repository (main branch)
repo_dir = Path('/content/MMDP-OD-RTDP-PROJECT')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print('Cloning repository from GitHub...')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/t-rays/MMDP-OD-RTDP-PROJECT.git', str(repo_dir)], check=True)

PROJECT_ROOT = repo_dir
OUTPUT_DIR = Path('/content/MMDP_OUTPUT')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / 'MMDP_results.csv'

print('Installing the mmdp package...')
# A regular (non-editable) install works immediately in the running kernel;
# an editable install would require a runtime restart before `import mmdp`.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', str(PROJECT_ROOT) + '[notebooks]'],
    check=True,
)
import importlib
importlib.invalidate_caches()

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CSV:', RESULTS_CSV)

In [ ]:
# Import the map visualization helper from the installed package
from functools import partial

from ipywidgets import interact, IntSlider
from mmdp.analysis.notebook_visualizer import plot_map_visualization

plot_map_visualization = partial(plot_map_visualization, maps_root=PROJECT_ROOT / 'maps')

In [ ]:
# Define the run and analysis functions
import subprocess, sys
import pandas as pd
from IPython.display import display, Image, Markdown

GROUP_MAPS = {
    'easy': ['empty-8-8'],
    'medium': ['warehouse-10-20-10-2-1'],
    'hard': ['room-64-64-16'],
}


def run_group(group):
    print('='*80)
    print(f"Starting the {group} experiment: {', '.join(GROUP_MAPS[group])}")
    print('24 conditions: 1 map x 6 agent counts x 2 seeds x 2 algorithms')
    print('='*80, flush=True)
    command = [
        sys.executable, '-u', str(PROJECT_ROOT/'scripts/run_compact_matrix.py'),
        '--group', group, '--output', str(RESULTS_CSV),
    ]
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Experiment failed with code {code}')


def comparison_table(group):
    df = pd.read_csv(RESULTS_CSV)
    part = df[(df['map_group'] == group) & (df['status'] == 'ok')].copy()
    metrics = ['planning_time_seconds','planning_peak_memory_delta_mb','states_examined','success_rate']
    for col in ['n_agents', *metrics]:
        part[col] = pd.to_numeric(part[col], errors='coerce')
    means = part.groupby(['n_agents','algorithm'], as_index=False)[metrics].mean()
    wide = means.pivot(index='n_agents', columns='algorithm', values=metrics)
    result = pd.DataFrame(index=wide.index)
    names = {
        'planning_time_seconds':'time_s',
        'planning_peak_memory_delta_mb':'memory_mb',
        'states_examined':'states',
        'success_rate':'success',
    }
    for metric, short in names.items():
        result[f'{short}_baseline'] = wide[(metric,'baseline')]
        result[f'{short}_od'] = wide[(metric,'od')]
        result[f'{short}_difference_OD_minus_Baseline'] = wide[(metric,'od')] - wide[(metric,'baseline')]
    return result.reset_index().round(4)


def analyze_group(group):
    graph_dir = OUTPUT_DIR/'graphs'/group
    command = [
        sys.executable, str(PROJECT_ROOT/'scripts/analyze_compact_results.py'),
        str(RESULTS_CSV), '--group', group, '--output-dir', str(graph_dir),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    display(Markdown('## Mean comparison table by number of agents'))
    display(comparison_table(group))
    figures = [
        ('01_time.png', 'Planning time - lower is better'),
        ('02_memory.png', 'Planning memory increase - lower is better'),
        ('03_states_and_success.png', 'States examined and success rate'),
    ]
    for filename, title in figures:
        display(Markdown(f'### {title}'))
        display(Image(filename=str(graph_dir/filename)))
    print('\nSingle CSV:', RESULTS_CSV)
    print('Graphs:', graph_dir)
    print('Nothing is downloaded automatically; use the Files panel to download manually.')


def run_and_analyze(group):
    run_group(group)
    analyze_group(group)

print('Preparation finished. Run one of the three difficulty cells below.')

## Easy map

Map: `empty-8-8`.

In [ ]:
def _plot_easy(num_agents):
    plot_map_visualization('empty-8-8', 'empty-8-8-bundled-1', num_agents)
interact(_plot_easy, num_agents=IntSlider(min=1, max=6, step=1, value=3, description='Agents:'))

In [ ]:
run_and_analyze("easy")

## Medium map

Map: `warehouse-10-20-10-2-1`.

In [ ]:
def _plot_med(num_agents):
    plot_map_visualization('warehouse-10-20-10-2-1', 'warehouse-10-20-10-2-1-bundled-1', num_agents)
interact(_plot_med, num_agents=IntSlider(min=1, max=6, step=1, value=4, description='Agents:'))

In [ ]:
run_and_analyze("medium")

## Hard map

Map: `room-64-64-16`.

In [ ]:
def _plot_hard(num_agents):
    plot_map_visualization('room-64-64-16', 'room-64-64-16-even-1', num_agents)
interact(_plot_hard, num_agents=IntSlider(min=1, max=6, step=1, value=6, description='Agents:'))

In [ ]:
run_and_analyze("hard")

## Output

The folder `/content/MMDP_OUTPUT` will contain:

- `MMDP_results.csv` - the single results file from every group that was run
- `graphs/easy` - three figures for the easy level
- `graphs/medium` - three figures for the medium level
- `graphs/hard` - three figures for the hard level

Re-running the same cell skips conditions that already exist in the CSV.